In [1]:
import os
from pathlib import Path

import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import InputLayer, Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [2]:
# =========================
# SETTINGS
# =========================
ROOT_DIR = Path(r"D:\Exams\ICT\dataset")
IMAGE_DIM = (224, 224)
BATCH = 32
TOTAL_EPOCHS = 10

SAVE_DIR = Path("D:\Exams\ICT\models")
SAVE_DIR.mkdir(exist_ok=True)

OUTPUT_MODEL = SAVE_DIR / "basic_cnn_weights.keras"

<>:9: SyntaxWarning: invalid escape sequence '\E'
<>:9: SyntaxWarning: invalid escape sequence '\E'
C:\Users\HP\AppData\Local\Temp\ipykernel_4696\1657189166.py:9: SyntaxWarning: invalid escape sequence '\E'
  SAVE_DIR = Path("D:\Exams\ICT\models")


In [3]:
# =========================
# CLASS SELECTION
# =========================
excluded_labels = {"J", "Z"}

selected_classes = [
    folder.name
    for folder in sorted(ROOT_DIR.iterdir())
    if folder.is_dir() and folder.name.upper() not in excluded_labels
]

print("Selected classes:", selected_classes)
print("Number of classes:", len(selected_classes))

Selected classes: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y']
Number of classes: 24


In [4]:
# =========================
# IMAGE LOADING PIPELINE
# =========================
image_loader = ImageDataGenerator(
    rescale=1.0 / 255.0,
    validation_split=0.20
)

train_data = image_loader.flow_from_directory(
    directory=str(ROOT_DIR),
    target_size=IMAGE_DIM,
    batch_size=BATCH,
    class_mode="categorical",
    classes=selected_classes,
    subset="training",
    shuffle=True,
    seed=42
)

valid_data = image_loader.flow_from_directory(
    directory=str(ROOT_DIR),
    target_size=IMAGE_DIM,
    batch_size=BATCH,
    class_mode="categorical",
    classes=selected_classes,
    subset="validation",
    shuffle=False,
    seed=42
)

num_output_classes = train_data.num_classes

print("Class mapping:", train_data.class_indices)

Found 3458 images belonging to 24 classes.


Found 863 images belonging to 24 classes.
Class mapping: {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4, 'F': 5, 'G': 6, 'H': 7, 'I': 8, 'K': 9, 'L': 10, 'M': 11, 'N': 12, 'O': 13, 'P': 14, 'Q': 15, 'R': 16, 'S': 17, 'T': 18, 'U': 19, 'V': 20, 'W': 21, 'X': 22, 'Y': 23}


In [5]:
# =========================
# CNN ARCHITECTURE
# =========================
cnn_model = Sequential(name="custom_basic_cnn")

cnn_model.add(InputLayer(input_shape=(IMAGE_DIM[0], IMAGE_DIM[1], 3)))

cnn_model.add(Conv2D(filters=32, kernel_size=(3, 3), activation="relu"))
cnn_model.add(MaxPooling2D(pool_size=(2, 2)))

cnn_model.add(Conv2D(filters=64, kernel_size=(3, 3), activation="relu"))
cnn_model.add(MaxPooling2D(pool_size=(2, 2)))

cnn_model.add(Conv2D(filters=128, kernel_size=(3, 3), activation="relu"))
cnn_model.add(MaxPooling2D(pool_size=(2, 2)))

cnn_model.add(Flatten())
cnn_model.add(Dense(256, activation="relu"))
cnn_model.add(Dropout(0.5))
cnn_model.add(Dense(num_output_classes, activation="softmax"))

d:\Exams\ICT\signEnv\Lib\site-packages\keras\src\layers\core\input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(


In [6]:
# =========================
# COMPILATION
# =========================
cnn_model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

cnn_model.summary()

Model: "custom_basic_cnn"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │    22,151,424 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 24)             │         6,168 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 22,250,840 (84.88 MB)

 Trainable params: 22,250,840 (84.88 MB)

 Non-trainable params: 0 (0.00 B)

In [7]:
# =========================
# MODEL TRAINING
# =========================
training_result = cnn_model.fit(
    train_data,
    validation_data=valid_data,
    epochs=TOTAL_EPOCHS
)

Epoch 1/10
109/109 ━━━━━━━━━━━━━━━━━━━━ 136s 1s/step - accuracy: 0.5130 - loss: 1.7765 - val_accuracy: 0.9224 - val_loss: 0.3528
Epoch 2/10
109/109 ━━━━━━━━━━━━━━━━━━━━ 113s 1s/step - accuracy: 0.8855 - loss: 0.3741 - val_accuracy: 0.9722 - val_loss: 0.1376
Epoch 3/10
109/109 ━━━━━━━━━━━━━━━━━━━━ 128s 1s/step - accuracy: 0.9381 - loss: 0.1961 - val_accuracy: 0.9710 - val_loss: 0.0991
Epoch 4/10
109/109 ━━━━━━━━━━━━━━━━━━━━ 125s 1s/step - accuracy: 0.9569 - loss: 0.1381 - val_accuracy: 0.9629 - val_loss: 0.1167
Epoch 5/10
109/109 ━━━━━━━━━━━━━━━━━━━━ 161s 1s/step - accuracy: 0.9653 - loss: 0.1052 - val_accuracy: 0.9780 - val_loss: 0.0639
Epoch 6/10
109/109 ━━━━━━━━━━━━━━━━━━━━ 142s 1s/step - accuracy: 0.9763 - loss: 0.0783 - val_accuracy: 0.9803 - val_loss: 0.0573
Epoch 7/10
109/109 ━━━━━━━━━━━━━━━━━━━━ 145s 1s/step - accuracy: 0.9780 - loss: 0.0650 - val_accuracy: 0.9838 - val_loss: 0.0490
Epoch 8/10
109/109 ━━━━━━━━━━━━━━━━━━━━ 148s 1s/step - accuracy: 0.9832 - loss: 0.0468 - val_accu

In [8]:
# =========================
# SAVE TRAINED MODEL
# =========================
cnn_model.save(OUTPUT_MODEL)

print(f"\n✅ Training complete. Model saved at: {OUTPUT_MODEL}")


✅ Training complete. Model saved at: D:\Exams\ICT\models\basic_cnn_weights.keras
